In [1]:
import pandas as pd

# the handover file from notebook 2: 81 features + target, no dates
df = pd.read_parquet('../data/processed/model_table.parquet')

# the same source notebook 2 started from, so we can recover the dates
raw = pd.read_parquet('../data/processed/sample_50k.parquet')

# repeat notebook 2's row filter exactly, so the rows line up
good = ['Fully Paid']
bad  = ['Charged Off', 'Default']
keep = raw['loan_status'].isin(good + bad)

labelled = raw[keep].copy().reset_index(drop=True)

print(f"model_table rows: {len(df)}")
print(f"refiltered rows : {len(labelled)}")

# position-matching is only valid if both sides are the same length
assert len(labelled) == len(df)

df['issue_d'] = labelled['issue_d']
df['earliest_cr_line'] = labelled['earliest_cr_line']

print(df[['issue_d', 'earliest_cr_line']].head())
print(df.shape)



model_table rows: 29619
refiltered rows : 29619
    issue_d earliest_cr_line
0  Dec-2015         Dec-2003
1  Dec-2015         Oct-1976
2  Dec-2015         Nov-2009
3  Dec-2015         Mar-2006
4  Dec-2015         Dec-1982
(29619, 84)


In [2]:
df['issue_d'].head(5)
# df['issue_d'].isna().sum()
df['issue_d'].isna().mean()

np.float64(0.0)

In [3]:
# turn the text date into a real date ("Dec-2015" -> 2015-12-01)
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

# the date with 80% of loans before it and the newest 20% after
cut = df['issue_d'].quantile(0.8)
print('cut-off date:', cut.date())

train = df[df['issue_d'] <  cut].copy()
test  = df[df['issue_d'] >= cut].copy()

print(f"train rows: {len(train):6d}   share: {len(train)/len(df):.1%}   bad rate: {train['target'].mean():.3%}")
print(f"test  rows: {len(test):6d}   share: {len(test)/len(df):.1%}   bad rate: {test['target'].mean():.3%}")

# nothing lost, nothing duplicated
assert len(train) + len(test) == len(df)
print('date range train:', train['issue_d'].min().date(), '->', train['issue_d'].max().date())
print('date range test :', test['issue_d'].min().date(), '->', test['issue_d'].max().date())

cut-off date: 2016-10-01
train rows:  23399   share: 79.0%   bad rate: 19.586%
test  rows:   6220   share: 21.0%   bad rate: 21.720%
date range train: 2007-07-01 -> 2016-09-01
date range test : 2016-10-01 -> 2018-12-01


In [4]:
by_year = df.groupby(df['issue_d'].dt.year).agg(
    rows        = ('target', 'size'),
    bad_rate    = ('target', 'mean'),
    share_60m   = ('term',   lambda s: (s == 60).mean()),
)
by_year['bad_rate']  = (by_year['bad_rate']  * 100).round(2)
by_year['share_60m'] = (by_year['share_60m'] * 100).round(1)
print(by_year)

# the same, split by term, for the test years only
print(
    df[df['issue_d'] >= '2016-10-01']
      .groupby([df['issue_d'].dt.year, 'term'])['target']
      .agg(['size', 'mean'])
)

         rows  bad_rate  share_60m
issue_d                           
2007        9     33.33        0.0
2008       32     15.62        0.0
2009      112     13.39        0.0
2010      269     10.04       31.2
2011      503     14.51       34.4
2012     1180     14.66       18.3
2013     2955     15.13       26.0
2014     4951     18.76       28.2
2015     8228     20.35       25.7
2016     6438     24.23       20.8
2017     3708     22.98       23.2
2018     1234     14.26       26.7
              size      mean
issue_d term                
2016    36.0  1004  0.225100
        60.0   274  0.354015
2017    36.0  2847  0.201616
        60.0   861  0.322880
2018    36.0   905  0.118232
        60.0   329  0.209726


In [5]:
# --- decision: restrict to 2013-2017 -------------------------------------
# 2018 : only 1,234 finished loans, and they are mostly early prepayers,
#        so their 14.3% bad rate is far below the truth.
# <2013: 153 loans before 2010, no 60-month product existed, crisis-era
#        lending rules. Different product, too few rows to learn from.
YEAR_MIN, YEAR_MAX = 2013, 2017

years = df['issue_d'].dt.year
df = df[(years >= YEAR_MIN) & (years <= YEAR_MAX)].copy().reset_index(drop=True)

print('rows after trim:', len(df))
print('bad rate       :', f"{df['target'].mean():.3%}")

# --- re-split on the trimmed data ----------------------------------------
cut = df['issue_d'].quantile(0.8)
print('cut-off date   :', cut.date())

train = df[df['issue_d'] <  cut].copy()
test  = df[df['issue_d'] >= cut].copy()

print(f"train rows: {len(train):6d}   share: {len(train)/len(df):.1%}   bad rate: {train['target'].mean():.3%}")
print(f"test  rows: {len(test):6d}   share: {len(test)/len(df):.1%}   bad rate: {test['target'].mean():.3%}")

assert len(train) + len(test) == len(df)
print('train:', train['issue_d'].min().date(), '->', train['issue_d'].max().date())
print('test :', test['issue_d'].min().date(), '->', test['issue_d'].max().date())

rows after trim: 26280
bad rate       : 20.784%
cut-off date   : 2016-09-01
train rows:  20902   share: 79.5%   bad rate: 20.046%
test  rows:   5378   share: 20.5%   bad rate: 23.652%
train: 2013-01-01 -> 2016-08-01
test : 2016-09-01 -> 2017-12-01


In [6]:
# assign it back, or nothing changes
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

# exact whole months, measured from issue_d - never from today
df['credit_history_months'] = (
    (df['issue_d'].dt.year  - df['earliest_cr_line'].dt.year) * 12
    + (df['issue_d'].dt.month - df['earliest_cr_line'].dt.month)
)

print('negative values:', (df['credit_history_months'] < 0).sum())
print(df['credit_history_months'].describe())

negative values: 0
count    26280.000000
mean       197.671918
std         90.690052
min         37.000000
25%        136.000000
50%        180.000000
75%        243.000000
max        781.000000
Name: credit_history_months, dtype: float64


In [7]:
bands = pd.qcut(df['credit_history_months'], 10, duplicates='drop')
print(df.groupby(bands, observed=True)['target'].agg(['mean', 'size']))

                           mean  size
credit_history_months                
(36.999, 100.0]        0.234224  2694
(100.0, 127.0]         0.235227  2640
(127.0, 144.0]         0.225770  2631
(144.0, 161.0]         0.213405  2596
(161.0, 180.0]         0.208534  2695
(180.0, 201.0]         0.205894  2545
(201.0, 228.0]         0.190440  2636
(228.0, 263.0]         0.197844  2598
(263.0, 322.0]         0.183790  2628
(322.0, 781.0]         0.182270  2617


In [8]:
print(df['earliest_cr_line'].min())
print((df['credit_history_months'] > 600).sum())   # over 50 years
print(df.nlargest(5, 'credit_history_months')[['earliest_cr_line', 'issue_d', 'credit_history_months']])

1952-01-01 00:00:00
24
      earliest_cr_line    issue_d  credit_history_months
19205       1952-01-01 2017-02-01                    781
8136        1957-01-01 2017-09-01                    728
19566       1959-05-01 2017-01-01                    692
4672        1958-07-01 2015-06-01                    683
13046       1959-03-01 2016-02-01                    683


In [9]:
CREDIT_HISTORY_CAP = 600   # 50 years - domain judgement, not read from the data
df['credit_history_months'] = df['credit_history_months'].clip(upper=CREDIT_HISTORY_CAP)

In [10]:
import numpy as np

df['loan_to_income']  = df['loan_amnt'] / df['annual_inc']
df['revol_to_income'] = df['revol_bal'] / df['annual_inc']

safe_total = df['total_acc'].replace(0, np.nan)
df['open_to_total_acc'] = df['open_acc'] / safe_total

new = ['loan_to_income', 'revol_to_income', 'open_to_total_acc']
print(df[new].describe().T)
print('\ninf values:', np.isinf(df[new]).sum().to_dict())
print('missing   :', df[new].isna().sum().to_dict())

                     count      mean       std       min       25%       50%  \
loan_to_income     26280.0  0.216257  0.169409  0.000436  0.125000  0.200000   
revol_to_income    26280.0  0.227474  0.224282  0.000000  0.100113  0.178729   
open_to_total_acc  26280.0  0.504475  0.175551  0.046154  0.375000  0.483871   

                        75%    max  
loan_to_income     0.291227  19.50  
revol_to_income    0.295309  10.76  
open_to_total_acc  0.615385   1.00  

inf values: {'loan_to_income': 0, 'revol_to_income': 0, 'open_to_total_acc': 0}
missing   : {'loan_to_income': 0, 'revol_to_income': 0, 'open_to_total_acc': 0}


In [11]:
for col in new:
    bands = pd.qcut(df[col], 10, duplicates='drop')
    print('\n---', col, '---')
    print(df.groupby(bands, observed=True)['target'].agg(['mean', 'size']))


--- loan_to_income ---
                         mean  size
loan_to_income                     
(-0.000564, 0.0755]  0.137590  2631
(0.0755, 0.111]      0.146332  2713
(0.111, 0.141]       0.176103  2561
(0.141, 0.169]       0.163218  2610
(0.169, 0.2]         0.185792  2928
(0.2, 0.232]         0.188817  2325
(0.232, 0.268]       0.227549  2628
(0.268, 0.316]       0.238204  2628
(0.316, 0.377]       0.287671  2628
(0.377, 19.5]        0.328387  2628

--- revol_to_income ---
                      mean  size
revol_to_income                 
(-0.001, 0.0507]  0.200533  2628
(0.0507, 0.085]   0.184551  2628
(0.085, 0.115]    0.200533  2628
(0.115, 0.146]    0.190639  2628
(0.146, 0.179]    0.203577  2628
(0.179, 0.216]    0.203196  2628
(0.216, 0.265]    0.219559  2628
(0.265, 0.333]    0.218417  2628
(0.333, 0.443]    0.221842  2628
(0.443, 10.76]    0.235540  2628

--- open_to_total_acc ---
                       mean  size
open_to_total_acc                
(0.0452, 0.292]    0.165781 

In [12]:
bands = pd.qcut(df['revol_util'], 10, duplicates='drop')
print(df.groupby(bands, observed=True)['target'].agg(['mean', 'size']))

                    mean  size
revol_util                    
(-0.001, 19.1]  0.168373  2637
(19.1, 29.7]    0.176021  2619
(29.7, 38.0]    0.187595  2628
(38.0, 45.4]    0.205177  2627
(45.4, 52.5]    0.204059  2661
(52.5, 59.4]    0.235384  2617
(59.4, 66.8]    0.215103  2622
(66.8, 74.8]    0.221286  2612
(74.8, 84.7]    0.230271  2623
(84.7, 162.0]   0.235137  2624


In [13]:
df = df.drop(columns='revol_to_income')

In [14]:
# # df.filter(like='mths_since')
# # [c for c in df.columns if 'mths_since' in c]
# # df['mths_since_last_delinq_missing'] = df['mths_since_last_delinq'].isna().astype('int8')
# df['mths_since_last_delinq'].isna().mean()


In [15]:
# which columns use "blank" to mean "never happened"?
since_cols = [c for c in df.columns if 'mths_since' in c]
print(df[since_cols].isna().mean().round(3).sort_values(ascending=False))

# build one flag and test the idea before building the rest
df['mths_since_last_delinq_missing'] = df['mths_since_last_delinq'].isna().astype('int8')

print()
print(df.groupby('mths_since_last_delinq_missing')['target'].agg(['mean', 'size']))

mths_since_last_record            0.816
mths_since_recent_bc_dlq          0.745
mths_since_last_major_derog       0.719
mths_since_recent_revol_delinq    0.647
mths_since_rcnt_il                0.611
mths_since_last_delinq            0.491
mths_since_recent_inq             0.097
mths_since_recent_bc              0.009
dtype: float64

                                    mean   size
mths_since_last_delinq_missing                 
0                               0.211510  13380
1                               0.204031  12900


In [16]:
# recency, among borrowers who actually have a delinquency on record
has_delinq = df[df['mths_since_last_delinq'].notna()]
bands = pd.qcut(has_delinq['mths_since_last_delinq'], 10, duplicates='drop')
print(has_delinq.groupby(bands, observed=True)['target'].agg(['mean', 'size']))

# all the flags at once, and whether any of them separate
print()
for col in since_cols:
    flag = df[col].isna().astype('int8')
    rates = df.groupby(flag)['target'].mean()
    if len(rates) == 2:
        print(f"{col:35s} missing={df[col].isna().mean():.1%}  "
              f"present={rates[0]:.3f}  blank={rates[1]:.3f}  gap={rates[0]-rates[1]:+.3f}")

                            mean  size
mths_since_last_delinq                
(-0.001, 7.0]           0.226456  1391
(7.0, 13.0]             0.222587  1523
(13.0, 18.0]            0.212073  1259
(18.0, 24.0]            0.207207  1332
(24.0, 30.0]            0.207772  1261
(30.0, 37.0]            0.195249  1347
(37.0, 45.0]            0.227206  1360
(45.0, 55.0]            0.205950  1311
(55.0, 67.0]            0.208138  1278
(67.0, 135.0]           0.199545  1318

mths_since_last_delinq              missing=49.1%  present=0.212  blank=0.204  gap=+0.007
mths_since_last_record              missing=81.6%  present=0.229  blank=0.203  gap=+0.026
mths_since_last_major_derog         missing=71.9%  present=0.220  blank=0.203  gap=+0.017
mths_since_rcnt_il                  missing=61.1%  present=0.236  blank=0.190  gap=+0.046
mths_since_recent_bc                missing=0.9%  present=0.208  blank=0.169  gap=+0.040
mths_since_recent_bc_dlq            missing=74.5%  present=0.209  blank=0.208  gap

In [17]:
# the two that earned their place
for col in ['mths_since_recent_inq', 'mths_since_rcnt_il']:
    df[col + '_missing'] = df[col].isna().astype('int8')

# the delinquency family carries nothing - flag and value both flat
dead = ['mths_since_last_delinq',
        'mths_since_recent_bc_dlq',
        'mths_since_recent_revol_delinq',
        'mths_since_last_major_derog',
        'mths_since_last_record',
        'mths_since_recent_bc']
df = df.drop(columns=dead + ['mths_since_last_delinq_missing'])

print('dropped:', len(dead) + 1)
print('shape  :', df.shape)

# does recency matter for inquiries, the way it didn't for delinquency?
has_inq = df[df['mths_since_recent_inq'].notna()]
bands = pd.qcut(has_inq['mths_since_recent_inq'], 10, duplicates='drop')
print(has_inq.groupby(bands, observed=True)['target'].agg(['mean', 'size']))

dropped: 7
shape  : (26280, 83)
                           mean  size
mths_since_recent_inq                
(-0.001, 1.0]          0.259071  4906
(1.0, 3.0]             0.225829  4220
(3.0, 4.0]             0.201799  1779
(4.0, 5.0]             0.197781  1532
(5.0, 7.0]             0.210920  2674
(7.0, 9.0]             0.199178  2189
(9.0, 11.0]            0.196606  1709
(11.0, 16.0]           0.189042  2756
(16.0, 24.0]           0.160425  1976


In [18]:
import numpy as np

the_columns = ['annual_inc', 'revol_bal', 'loan_amnt', 'tot_cur_bal', 'total_bal_il']

# skew check: mean well above median = long right tail
check = pd.DataFrame({
    'mean':   df[the_columns].mean(),
    'median': df[the_columns].median(),
    'max':    df[the_columns].max(),
    'skew':   df[the_columns].skew(),
    'missing': df[the_columns].isna().mean(),
})
print(check.sort_values('skew', ascending=False).round(2))

# log1p is undefined for negatives - check before, don't find out after
print('\nnegative values:', (df[the_columns] < 0).sum().to_dict())

# log1p = log(1 + x), so zeros are safe (plain log(0) is -inf)
log_df = np.log1p(df[the_columns]).add_prefix('log_')
df = pd.concat([df, log_df], axis=1)

print('\nshape:', df.shape)
print(df[['annual_inc', 'log_annual_inc', 'tot_cur_bal', 'log_tot_cur_bal']].describe().round(2).T)

                   mean   median        max   skew  missing
annual_inc     76202.78  65000.0  6000000.0  34.41      0.0
revol_bal      16254.81  11134.5   678330.0   7.98      0.0
total_bal_il   35063.03  23189.0   510553.0   3.14      0.6
tot_cur_bal   140218.73  80459.5  2471713.0   2.54      0.0
loan_amnt      14498.52  12000.0    40000.0   0.77      0.0

negative values: {'annual_inc': 0, 'revol_bal': 0, 'loan_amnt': 0, 'tot_cur_bal': 0, 'total_bal_il': 0}

shape: (26280, 88)


                   count       mean        std      min       25%       50%  \
annual_inc       26280.0   76202.78   67508.96  1000.00  46000.00  65000.00   
log_annual_inc   26280.0      11.09       0.53     6.91     10.74     11.08   
tot_cur_bal      26280.0  140218.73  154375.63     0.00  29412.50  80459.50   
log_tot_cur_bal  26280.0      11.21       1.27     0.00     10.29     11.30   

                       75%         max  
annual_inc        90000.00  6000000.00  
log_annual_inc       11.41       15.61  
tot_cur_bal      210620.00  2471713.00  
log_tot_cur_bal      12.26       14.72  


In [19]:
df = df.drop(columns='log_loan_amnt')

In [20]:
# does missingness track time? if yes, the column is a date in disguise
il_family = [c for c in df.columns if '_il' in c]
print('_il family:', il_family)
print()
print(df.groupby(df['issue_d'].dt.year)[il_family].apply(lambda g: g.isna().mean()).round(2))

# and: does the flag still separate INSIDE a single year?
one_year = df[df['issue_d'].dt.year == 2016]
print()
print(one_year.groupby('mths_since_rcnt_il_missing')['target'].agg(['mean', 'size']))

_il family: ['open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'mo_sin_old_il_acct', 'num_il_tl', 'total_il_high_credit_limit', 'mths_since_rcnt_il_missing', 'log_total_bal_il']

         open_act_il  open_il_12m  open_il_24m  mths_since_rcnt_il  \
issue_d                                                              
2013            1.00         1.00         1.00                1.00   
2014            1.00         1.00         1.00                1.00   
2015            0.95         0.95         0.95                0.95   
2016            0.00         0.00         0.00                0.03   
2017            0.00         0.00         0.00                0.03   

         total_bal_il  mo_sin_old_il_acct  num_il_tl  \
issue_d                                                
2013             1.00                0.03        0.0   
2014             1.00                0.03        0.0   
2015             0.95                0.03        0.0   
2016             

In [21]:
# how much does each column's missing rate move across years?
miss_by_year = df.groupby(df['issue_d'].dt.year).apply(lambda g: g.isna().mean())
spread = (miss_by_year.max() - miss_by_year.min()).sort_values(ascending=False)

print('columns whose missingness jumps by more than 20 points across years:')
print(spread[spread > 0.20].round(2))

columns whose missingness jumps by more than 20 points across years:
open_rv_24m           1.00
max_bal_bc            1.00
total_bal_il          1.00
open_il_12m           1.00
open_il_24m           1.00
open_acc_6m           1.00
open_act_il           1.00
open_rv_12m           1.00
all_util              1.00
inq_fi                1.00
log_total_bal_il      1.00
inq_last_12m          1.00
total_cu_tl           1.00
mths_since_rcnt_il    0.97
il_util               0.86
dtype: float64


In [22]:
KEEP = ['emp_length']          # real feature - drifts over time, but not a schema change
time_dependent = [c for c in spread[spread > 0.20].index if c not in KEEP]
also_drop = ['mths_since_rcnt_il_missing', 'log_total_bal_il', 'log_loan_amnt']
to_drop = [c for c in time_dependent + also_drop if c in df.columns]

df = df.drop(columns=to_drop)
print('dropped', len(to_drop), 'columns:')
print(to_drop)
print('shape:', df.shape)

dropped 17 columns:
['open_rv_24m', 'max_bal_bc', 'total_bal_il', 'open_il_12m', 'open_il_24m', 'open_acc_6m', 'open_act_il', 'open_rv_12m', 'all_util', 'inq_fi', 'log_total_bal_il', 'inq_last_12m', 'total_cu_tl', 'mths_since_rcnt_il', 'il_util', 'mths_since_rcnt_il_missing', 'log_total_bal_il']
shape: (26280, 71)


In [23]:
emp = labelled[['emp_length', 'issue_d']].copy()
emp['year'] = pd.to_datetime(emp['issue_d'], format='%b-%Y').dt.year
emp['blank'] = emp['emp_length'].isna() | (emp['emp_length'] == 'n/a')
print(emp.groupby('year')['blank'].mean().round(3))
print()
print(labelled['emp_length'].value_counts(dropna=False).head(12))

year
2007    0.000
2008    0.000
2009    0.000
2010    0.022
2011    0.038
2012    0.036
2013    0.051
2014    0.052
2015    0.056
2016    0.063
2017    0.071
2018    0.094
Name: blank, dtype: float64

emp_length
10+ years    9818
2 years      2652
< 1 year     2419
3 years      2301
1 year       1961
5 years      1839
4 years      1784
NaN          1718
6 years      1364
8 years      1314
7 years      1303
9 years      1146
Name: count, dtype: int64


In [24]:
print(df['emp_length'].dtype)
print(df['emp_length'].isna().mean().round(3))
print(df.groupby(df['issue_d'].dt.year)['emp_length'].apply(lambda s: s.isna().mean()).round(3))

float64
0.058
issue_d
2013    0.051
2014    0.052
2015    0.056
2016    0.063
2017    0.071
Name: emp_length, dtype: float64


In [25]:
since_cols = [c for c in df.columns if 'mths_since' in c]
print(df[since_cols].isna().mean().round(3).sort_values(ascending=False))
print()

# does "blank" separate good from bad, for each of them?
for col in since_cols:
    flag = df[col].isna().astype('int8')
    rates = df.groupby(flag)['target'].mean()
    if len(rates) == 2:
        print(f"{col:35s} missing={df[col].isna().mean():.1%}  "
              f"present={rates[0]:.3f}  blank={rates[1]:.3f}  gap={rates[0]-rates[1]:+.3f}")

mths_since_recent_inq            0.097
mths_since_recent_inq_missing    0.000
dtype: float64

mths_since_recent_inq               missing=9.7%  present=0.213  blank=0.158  gap=+0.055


In [26]:
print(df.shape)

(26280, 71)


In [27]:
print(len(df.columns))
print(sorted(df.columns.tolist()))

71
['acc_now_delinq', 'acc_open_past_24mths', 'addr_state', 'annual_inc', 'application_type', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'credit_history_months', 'delinq_2yrs', 'delinq_amnt', 'disbursement_method', 'dti', 'earliest_cr_line', 'emp_length', 'fico_range_high', 'fico_range_low', 'home_ownership', 'initial_list_status', 'inq_last_6mths', 'issue_d', 'loan_amnt', 'loan_to_income', 'log_annual_inc', 'log_revol_bal', 'log_tot_cur_bal', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_inq', 'mths_since_recent_inq_missing', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'open_acc', 'open_to_total_acc', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec', 

In [28]:
print('emp_length missing:', df['emp_length'].isna().mean().round(3))
print(df.groupby(df['issue_d'].dt.year)['emp_length'].apply(lambda s: s.isna().mean()).round(3))

emp_length missing: 0.058
issue_d
2013    0.051
2014    0.052
2015    0.056
2016    0.063
2017    0.071
Name: emp_length, dtype: float64


In [29]:
not_number_cols = ['purpose', 'home_ownership', 'verification_status', 'application_type','initial_list_status', 'disbursement_method', 'addr_state', 'zip_code']

for c in not_number_cols:
    if df[c].nunique() < 15:
        print('---', c, '---')
        t = df.groupby(c, observed=True)['target'].agg(['mean', 'size'])
        t['share'] = (t['size'] / len(df) * 100).round(1)
        print(t.sort_values('size', ascending=False).round(3))
        print()



--- purpose ---
                     mean   size  share
purpose                                
debt_consolidation  0.216  15417   58.7
credit_card         0.183   5874   22.4
home_improvement    0.199   1689    6.4
other               0.212   1531    5.8
major_purchase      0.194    526    2.0
medical             0.232    285    1.1
small_business      0.325    240    0.9
car                 0.156    231    0.9
moving              0.283    173    0.7
vacation            0.186    161    0.6
house               0.265    117    0.4
renewable_energy    0.250     24    0.1
wedding             0.167     12    0.0

--- home_ownership ---
                 mean   size  share
home_ownership                     
MORTGAGE        0.178  13094   49.8
RENT            0.242  10273   39.1
OWN             0.222   2911   11.1
ANY             0.500      2    0.0

--- verification_status ---
                      mean   size  share
verification_status                     
Source Verified      0.218  10576

In [30]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')
from src.columns import (CAT_ONE_HOT, CAT_FOLD_RARE, CAT_DROP,
                         CAT_FAIRNESS_ONLY, NON_FEATURES)

# are these two columns saying the same thing?
print((df['fico_range_high'] - df['fico_range_low']).value_counts())
    

4.0    26276
5.0        4
Name: count, dtype: int64


In [31]:
# two columns, one fact (always 4 apart) -> keep one number
df['fico'] = (df['fico_range_low'] + df['fico_range_high']) / 2
df = df.drop(columns=['fico_range_low', 'fico_range_high'])

# categories with no usable signal
df = df.drop(columns=[c for c in CAT_DROP if c in df.columns])

# levels too small to learn from, folded into an existing level
for col, (rare, into) in CAT_FOLD_RARE.items():
    before = df[col].nunique()
    df[col] = df[col].where(~df[col].isin(rare), into)
    print(f'{col}: {before} -> {df[col].nunique()} levels (folded into {into})')

print('shape:', df.shape)

purpose: 13 -> 8 levels (folded into other)
home_ownership: 4 -> 3 levels (folded into MORTGAGE)
shape: (26280, 68)


In [32]:
from src.features import (add_row_features, add_logs, drop_decided_columns,
                          fold_rare_levels, restrict_years, split_by_time,
                          fit_fill_values, apply_fills, encode_categories)

train, test, cut = split_by_time(df)
print('cut-off:', cut.date())

print(f"train: {len(train):6d}  bad rate {train['target'].mean():.3%}  "
      f"{train['issue_d'].min().date()} -> {train['issue_d'].max().date()}")
print(f"test : {len(test):6d}  bad rate {test['target'].mean():.3%}  "
      f"{test['issue_d'].min().date()} -> {test['issue_d'].max().date()}")

assert len(train) + len(test) == len(df)
assert train['issue_d'].is_monotonic_increasing
assert test['issue_d'].is_monotonic_increasing

cut-off: 2016-09-01
train:  20902  bad rate 20.046%  2013-01-01 -> 2016-08-01
test :   5378  bad rate 23.652%  2016-09-01 -> 2017-12-01


In [33]:
# judgement fill: blanks mean "no recent credit-seeking", which behaves like
# "longest ago", so they belong at the safe end - not at the median.
INQ_FILL = 36
train['mths_since_recent_inq'] = train['mths_since_recent_inq'].fillna(INQ_FILL)
test['mths_since_recent_inq']  = test['mths_since_recent_inq'].fillna(INQ_FILL)

# statistical fill: medians learned from TRAIN ONLY, applied unchanged to both
num_cols = train.select_dtypes(include='number').columns.drop('target')
fill_values = train[num_cols].median()

train[num_cols] = train[num_cols].fillna(fill_values)
test[num_cols]  = test[num_cols].fillna(fill_values)

print('train missing:', train.isna().sum().sum())
print('test  missing:', test.isna().sum().sum())
print('\nsome of the train medians used:')
print(fill_values.head())

train missing: 0
test  missing: 0

some of the train medians used:
loan_amnt     12600.00
term             36.00
emp_length        7.00
annual_inc    65000.00
dti              17.93
dtype: float64


In [34]:
# fix each column's level list from TRAIN, so both halves produce exactly
# the same dummy columns, in the same order
for col in CAT_ONE_HOT:
    cats = sorted(train[col].dropna().unique())
    train[col] = pd.Categorical(train[col], categories=cats)
    test[col]  = pd.Categorical(test[col],  categories=cats)

train = pd.get_dummies(train, columns=CAT_ONE_HOT, drop_first=True, dtype='int8')
test  = pd.get_dummies(test,  columns=CAT_ONE_HOT, drop_first=True, dtype='int8')

print('train:', train.shape)
print('test :', test.shape)
print('same columns, same order:', list(train.columns) == list(test.columns))

dummies = [c for c in train.columns if any(c.startswith(p + '_') for p in CAT_ONE_HOT)]
print(f'\n{len(dummies)} dummy columns:')
print(dummies)

train: (20902, 76)
test : (5378, 76)
same columns, same order: True

12 dummy columns:
['purpose_credit_card', 'purpose_debt_consolidation', 'purpose_home_improvement', 'purpose_major_purchase', 'purpose_medical', 'purpose_other', 'purpose_small_business', 'home_ownership_OWN', 'home_ownership_RENT', 'verification_status_Source Verified', 'verification_status_Verified', 'application_type_Joint App']


In [35]:
import numpy as np
from pathlib import Path

assert list(train.columns) == list(test.columns), "column sets differ"
assert train.isna().sum().sum() == 0, "missing values left in train"
assert test.isna().sum().sum()  == 0, "missing values left in test"

num = train.select_dtypes(include='number')
assert np.isinf(num).sum().sum() == 0, "inf values in train"

out = Path('../data/processed')
train.to_parquet(out / 'features_train.parquet', index=False)
test.to_parquet(out  / 'features_test.parquet',  index=False)

features = [c for c in train.columns if c not in NON_FEATURES]
print('train:', train.shape, ' test:', test.shape)
print('model features:', len(features))
print('bad rate  train:', round(train['target'].mean(), 4),
      ' test:', round(test['target'].mean(), 4))

train: (20902, 76)  test: (5378, 76)
model features: 71
bad rate  train: 0.2005  test: 0.2365


In [36]:
print(to_drop)

['open_rv_24m', 'max_bal_bc', 'total_bal_il', 'open_il_12m', 'open_il_24m', 'open_acc_6m', 'open_act_il', 'open_rv_12m', 'all_util', 'inq_fi', 'log_total_bal_il', 'inq_last_12m', 'total_cu_tl', 'mths_since_rcnt_il', 'il_util', 'mths_since_rcnt_il_missing', 'log_total_bal_il']


In [37]:


# rebuild from scratch using only src/, then compare to what the notebook made
check = pd.read_parquet('../data/processed/model_table.parquet')
src_raw = pd.read_parquet('../data/processed/sample_50k.parquet')
lab = src_raw[src_raw['loan_status'].isin(good + bad)].copy().reset_index(drop=True)
check['issue_d'] = pd.to_datetime(lab['issue_d'], format='%b-%Y')
check['earliest_cr_line'] = pd.to_datetime(lab['earliest_cr_line'], format='%b-%Y')

check = restrict_years(check)
check = add_row_features(check)
check = drop_decided_columns(check)
check = add_logs(check)
check = fold_rare_levels(check)

c_train, c_test, _ = split_by_time(check)
fv = fit_fill_values(c_train)
c_train, c_test = apply_fills(c_train, fv), apply_fills(c_test, fv)
c_train, c_test = encode_categories(c_train, c_test)

print('shapes match:', c_train.shape == train.shape and c_test.shape == test.shape)
print('columns match:', sorted(c_train.columns) == sorted(train.columns))

shapes match: True
columns match: True


In [38]:
import pandas as pd

for name, path in [("train", "../data/processed/features_train.parquet"),
                   ("test",  "../data/processed/features_test.parquet")]:
    df = pd.read_parquet(path)
    print(f"--- {name} ---")
    print("rows:        ", len(df))
    print("bad rate:    ", round(df['target'].mean(), 4))
    print("in time order:", df['issue_d'].is_monotonic_increasing)
    print("date range:  ", df['issue_d'].min().date(), "->", df['issue_d'].max().date())
    print()

--- train ---
rows:         20902
bad rate:     0.2005
in time order: True
date range:   2013-01-01 -> 2016-08-01

--- test ---
rows:         5378
bad rate:     0.2365
in time order: True
date range:   2016-09-01 -> 2017-12-01

